# LG Aimers HGB Top20 Experiment

고정 350 iteration HGB에서 2024 permutation importance 상위 20개 피처만 사용합니다.
기존 V2 Full 결과의 Trackman cache를 재사용하므로 Trackman 전체 재집계를 피합니다.


In [ ]:
from pathlib import Path

if Path("/content/drive/MyDrive").is_dir():
    DRIVE_ROOT = Path("/content/drive/MyDrive")
else:
    from google.colab import drive
    MOUNT_POINT = "/content/gdrive_hgb_v2"
    drive.mount(MOUNT_POINT)
    DRIVE_ROOT = Path(MOUNT_POINT) / "MyDrive"

print("DRIVE_ROOT =", DRIVE_ROOT)


In [ ]:
import shutil, subprocess

REPO_DIR = Path("/content/lg_aimers_experiment_lab_v2")
BRANCH = "agent/hgb-feature-selection-v2"

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run([
    "git", "clone", "--depth", "1", "--branch", BRANCH,
    "https://github.com/tswaincae1221/lg_aimers_experiment_lab.git",
    str(REPO_DIR),
], check=True)

subprocess.run([
    "python", "-m", "pip", "install", "-q", "-r",
    str(REPO_DIR / "requirements.txt")
], check=True)


In [ ]:
def resolve_drive_file(filename: str, preferred_folder: str = "aimers_data") -> Path:
    preferred = DRIVE_ROOT / preferred_folder / filename
    if preferred.is_file():
        return preferred
    matches = sorted(p for p in DRIVE_ROOT.rglob(filename) if p.is_file())
    if len(matches) == 1:
        return matches[0]
    raise RuntimeError(f"{filename}: candidates={matches[:20]}")

TRAIN_PATH = resolve_drive_file("train.csv")
TRACKMAN_PATH = resolve_drive_file("trackman_history.csv")
MAPPING_PATH = REPO_DIR / "resources" / "pitcher_trackman_mapping.csv"

FULL_RESULTS_DIR = DRIVE_ROOT / "aimers_data" / "results" / "hgb_feature_selection_v2_full"
TOP20_OUTPUT_DIR = DRIVE_ROOT / "aimers_data" / "results" / "hgb_top20_fixed350"
TOP20_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("TRAIN =", TRAIN_PATH)
print("TRACKMAN =", TRACKMAN_PATH)
print("FULL_RESULTS_DIR =", FULL_RESULTS_DIR)
print("TOP20_OUTPUT_DIR =", TOP20_OUTPUT_DIR)


In [ ]:
import subprocess, sys

cmd = [
    sys.executable, "-m", "src.hgb_top20_experiment",
    "--train", str(TRAIN_PATH),
    "--trackman", str(TRACKMAN_PATH),
    "--mapping", str(MAPPING_PATH),
    "--output-dir", str(TOP20_OUTPUT_DIR),
    "--reference-results-dir", str(FULL_RESULTS_DIR),
    "--trackman-cache-dir", str(FULL_RESULTS_DIR),
    "--validation-season", "2024",
]

print(" ".join(cmd))
subprocess.run(cmd, cwd=REPO_DIR, check=True)


In [ ]:
import json, pandas as pd

scores = pd.read_csv(TOP20_OUTPUT_DIR / "model_scores_top20.csv")
display(scores)

print("\nTop20 features:")
print((TOP20_OUTPUT_DIR / "top20_features.txt").read_text(encoding="utf-8"))

print("\nSummary:")
print(json.dumps(
    json.load(open(TOP20_OUTPUT_DIR / "run_summary_top20.json", encoding="utf-8")),
    ensure_ascii=False, indent=2
))
